# Synthetic Data Pipeline (notebook-local implementation)

This section demonstrates the **generic synthetic data pipeline** implementation  step by step, so you can present the flow:

1. Define all helper functions (DB connection, schema discovery, profiling, ordering,
   value generation, insert, and FK validation).
2. Configure connection + environment using `.env`.
3. Discover the database schema generically from `information_schema`.
4. Optionally profile existing data and/or sample CSVs (ranges, categories, lengths).
5. Plan FK-safe table order and truncate tables.
6. Run the **batched** synthetic pipeline (generate + insert) using Faker + profiles.
7. Validate foreign keys and summarize row counts.

All steps are **schema-agnostic** and driven only by DB metadata and `.env` settings.

### Helper function overview

The next code cells define reusable helpers:

- **Imports & Faker setup** – brings in standard libraries, `psycopg2`, `pandas`, and creates a shared `Faker` instance.
- **DBConfig & get_connection** – small wrapper around PostgreSQL connection details using `RealDictCursor`.
- **Schema discovery** – `fetch_db_schema` reads `information_schema` and builds a generic table/column/constraint model.
- **Profiling helpers** – `profile_existing_data` and `profile_sample_data` compute simple stats (ranges, categories, text lengths).
- **Ordering & truncate** – `plan_table_order` computes FK-safe order; `truncate_all_synthetic_tables` clears data safely.
- **Value generation** – `_generate_scalar_value` uses Faker + optional profiles to emit realistic values per column type.
- **Insert & validation** – `insert_synthetic_dataset` writes rows in FK-safe order;
  `validate_referential_integrity` checks for orphan rows.
- **Batched runner** – `run_synthetic_pipeline_batched` ties everything together in a memory-efficient loop.

In [ ]:
# Shared imports and Faker setup
from dataclasses import dataclass
from typing import Dict, Any, List, Optional, Set
from pathlib import Path
from decimal import Decimal
import uuid
import random
import math

import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
import pandas as pd
from faker import Faker

fake = Faker()

In [ ]:
# DB config and schema discovery helpers (aligned with src.synthetic_pipeline)
import os
from dataclasses import dataclass
from typing import Optional, Dict, Any, List
from contextlib import contextmanager

from psycopg2.extras import RealDictCursor
from sqlalchemy import create_engine
from sqlalchemy.engine import Engine
from dotenv import load_dotenv

# Load environment variables from .env if present
load_dotenv()


@dataclass
class DBConfig:
    """Basic Postgres-style settings, mainly for fallback.

    In normal use you just set DB_CONFIG in .env to any SQLAlchemy URL
    and call get_connection() with no arguments. DBConfig only matters
    if DB_CONFIG is missing.
    """

    host: str = "localhost"
    port: int = 5432
    dbname: str = "jdf"
    user: str = "postgres"
    password: str = "postgres"


def _resolve_url(config: Optional[DBConfig] = None) -> str:
    """Resolve a SQLAlchemy URL from env or DBConfig.

    Priority:
    1) DB_CONFIG from environment / .env (generic, any database)
    2) Build a Postgres URL from DBConfig (fallback)
    """

    env_url = os.getenv("DB_CONFIG")
    if env_url:
        # be tolerant of accidental surrounding quotes in .env
        return env_url.strip().strip("\"").strip("'")

    cfg = config or DBConfig()
    return (
        f"postgresql+psycopg2://{cfg.user}:{cfg.password}"
        f"@{cfg.host}:{cfg.port}/{cfg.dbname}",
    )


_ENGINE_CACHE: Dict[str, Engine] = {}


def _get_engine(url: str) -> Engine:
    """Return a cached SQLAlchemy Engine for the given URL."""

    engine = _ENGINE_CACHE.get(url)
    if engine is None:
        engine = create_engine(url)
        _ENGINE_CACHE[url] = engine
    return engine


@contextmanager
def get_connection(config: Optional[DBConfig] = None):
    """Yield a DBAPI connection created via SQLAlchemy.

    Usage everywhere in this notebook:

        with get_connection(cfg) as conn:
            cur = conn.cursor()
    """

    url = _resolve_url(config)
    engine = _get_engine(url)
    raw_conn = engine.raw_connection()

    # For psycopg2/Postgres URLs, wrap the connection so that
    # conn.cursor() returns a RealDictCursor, which the rest of the
    # notebook expects (row["col_name"] access).
    if url.startswith("postgresql+psycopg2://"):
        class _RealDictConnection:
            def __init__(self, conn):
                self._conn = conn

            def cursor(self, *args, **kwargs):
                kwargs.setdefault("cursor_factory", RealDictCursor)
                return self._conn.cursor(*args, **kwargs)

            def __getattr__(self, name):  # delegate commit, rollback, etc.
                return getattr(self._conn, name)

        conn = _RealDictConnection(raw_conn)
    else:
        conn = raw_conn

    try:
        yield conn
    finally:
        raw_conn.close()


def _table_id(t: Dict[str, Any]) -> str:
    return f"{t['schema']}.{t['table']}"


def fetch_db_schema(config: DBConfig) -> List[Dict[str, Any]]:
    """Return schema metadata for all non-system tables in the database.

    Matches the structure used by the library: a list of tables,
    each with "schema", "table", "columns", and "constraints" (PK/FK/UNIQUE/CHECK).
    """
    from collections import defaultdict

    tables: Dict[tuple, Dict[str, Any]] = {}

    with get_connection(config) as conn:
        with conn.cursor() as cur:
            # Columns
            cur.execute(
                """
                SELECT
                    table_schema,
                    table_name,
                    column_name,
                    ordinal_position,
                    data_type,
                    udt_name,
                    is_nullable,
                    column_default,
                    character_maximum_length,
                    numeric_precision,
                    numeric_scale
                FROM information_schema.columns
                WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
                  AND table_schema NOT LIKE 'pg_toast%'
                ORDER BY table_schema, table_name, ordinal_position
                """
            )
            for row in cur.fetchall():
                key = (row["table_schema"], row["table_name"])
                if key not in tables:
                    tables[key] = {
                        "schema": row["table_schema"],
                        "table": row["table_name"],
                        "columns": [],
                        "constraints": {
                            "primary_keys": [],
                            "foreign_keys": [],
                            "uniques": [],
                            "checks": [],
                        },
                    }
                tables[key]["columns"].append(
                    {
                        "name": row["column_name"],
                        "data_type": row["data_type"],
                        "udt_name": row["udt_name"],
                        "is_nullable": row["is_nullable"] == "YES",
                        "default": row["column_default"],
                        "max_length": row["character_maximum_length"],
                        "numeric_precision": row["numeric_precision"],
                        "numeric_scale": row["numeric_scale"],
                        "ordinal_position": row["ordinal_position"],
                    },
                )

            # Primary keys
            cur.execute(
                """
                SELECT
                    kcu.table_schema,
                    kcu.table_name,
                    tco.constraint_name,
                    kcu.column_name
                FROM information_schema.table_constraints tco
                JOIN information_schema.key_column_usage kcu
                  ON kcu.constraint_name = tco.constraint_name
                 AND kcu.constraint_schema = tco.constraint_schema
                WHERE tco.constraint_type = 'PRIMARY KEY'
                ORDER BY kcu.table_schema, kcu.table_name, tco.constraint_name, kcu.ordinal_position
                """
            )
            pk_map: Dict[tuple, Dict[str, Any]] = defaultdict(lambda: {"name": None, "columns": []})
            for row in cur.fetchall():
                key = (row["table_schema"], row["table_name"], row["constraint_name"])
                pk = pk_map[key]
                pk["name"] = row["constraint_name"]
                pk["columns"].append(row["column_name"])
            for (schema, table, _), pk in pk_map.items():
                tkey = (schema, table)
                if tkey in tables:
                    tables[tkey]["constraints"]["primary_keys"].append(pk)

            # Foreign keys (FK columns correctly aligned with referenced columns)
            cur.execute(
                """
                SELECT
                    tc.constraint_name,
                    kcu.table_schema,
                    kcu.table_name,
                    kcu.column_name,
                    ccu.table_schema AS foreign_table_schema,
                    ccu.table_name AS foreign_table_name,
                    ccu.column_name AS foreign_column_name
                FROM information_schema.table_constraints AS tc
                JOIN information_schema.key_column_usage AS kcu
                  ON tc.constraint_name = kcu.constraint_name
                 AND tc.constraint_schema = kcu.constraint_schema
                JOIN information_schema.referential_constraints AS rc
                  ON rc.constraint_name = tc.constraint_name
                 AND rc.constraint_schema = tc.constraint_schema
                JOIN information_schema.key_column_usage AS ccu
                  ON ccu.constraint_name = rc.unique_constraint_name
                 AND ccu.constraint_schema = rc.unique_constraint_schema
                 AND ccu.ordinal_position = kcu.position_in_unique_constraint
                WHERE tc.constraint_type = 'FOREIGN KEY'
                ORDER BY kcu.table_schema, kcu.table_name, tc.constraint_name, kcu.ordinal_position
                """
            )
            fk_map: Dict[tuple, Dict[str, Any]] = defaultdict(
                lambda: {
                    "name": None,
                    "columns": [],
                    "references": {"schema": None, "table": None, "columns": []},
                },
            )
            for row in cur.fetchall():
                key = (row["table_schema"], row["table_name"], row["constraint_name"])
                fk = fk_map[key]
                fk["name"] = row["constraint_name"]
                fk["columns"].append(row["column_name"])
                fk["references"]["schema"] = row["foreign_table_schema"]
                fk["references"]["table"] = row["foreign_table_name"]
                fk["references"]["columns"].append(row["foreign_column_name"])
            for (schema, table, _), fk in fk_map.items():
                tkey = (schema, table)
                if tkey in tables:
                    tables[tkey]["constraints"]["foreign_keys"].append(fk)

            # Unique constraints
            cur.execute(
                """
                SELECT
                    kcu.table_schema,
                    kcu.table_name,
                    tco.constraint_name,
                    kcu.column_name
                FROM information_schema.table_constraints tco
                JOIN information_schema.key_column_usage kcu
                  ON kcu.constraint_name = tco.constraint_name
                 AND kcu.constraint_schema = tco.constraint_schema
                WHERE tco.constraint_type = 'UNIQUE'
                ORDER BY kcu.table_schema, kcu.table_name, tco.constraint_name, kcu.ordinal_position
                """
            )
            uniq_map: Dict[tuple, Dict[str, Any]] = defaultdict(lambda: {"name": None, "columns": []})
            for row in cur.fetchall():
                key = (row["table_schema"], row["table_name"], row["constraint_name"])
                uq = uniq_map[key]
                uq["name"] = row["constraint_name"]
                uq["columns"].append(row["column_name"])
            for (schema, table, _), uq in uniq_map.items():
                tkey = (schema, table)
                if tkey in tables:
                    tables[tkey]["constraints"]["uniques"].append(uq)

            # Check constraints
            cur.execute(
                """
                SELECT
                    tc.table_schema,
                    tc.table_name,
                    tc.constraint_name,
                    cc.check_clause
                FROM information_schema.table_constraints tc
                JOIN information_schema.check_constraints cc
                  ON tc.constraint_name = cc.constraint_name
                 AND tc.constraint_schema = cc.constraint_schema
                WHERE tc.constraint_type = 'CHECK'
                ORDER BY tc.table_schema, tc.table_name, tc.constraint_name
                """
            )
            for row in cur.fetchall():
                key = (row["table_schema"], row["table_name"])
                if key in tables:
                    tables[key]["constraints"]["checks"].append(
                        {
                            "name": row["constraint_name"],
                            "clause": row["check_clause"],
                        },
                    )

    return [tables[key] for key in sorted(tables.keys())]

In [ ]:
# Profiling helpers for existing DB data and sample CSVs
def profile_existing_data(
    config: DBConfig,
    db_schema_list: List[Dict[str, Any]],
    max_categories: int = 50,
 ) -> Dict[str, Dict[str, Dict[str, Any]]]:
    """Profile existing data per column using live database rows.

    Returns profiles["schema.table"]["column_name"] with simple stats:
      - numeric: min, max
      - datetime: min, max
      - categorical: up to max_categories distinct values + frequencies
      - text: min_length, max_length
    """
    profiles: Dict[str, Dict[str, Dict[str, Any]]] = {}
    with get_connection(config) as conn:
        with conn.cursor() as cur:
            for t in db_schema_list:
                tid = _table_id(t)
                table_profiles: Dict[str, Dict[str, Any]] = {}
                profiles[tid] = table_profiles

                for col in t["columns"]:
                    col_name = col["name"]
                    data_type = (col["data_type"] or "").lower()
                    udt_name = (col["udt_name"] or "").lower()
                    full_col = f'"{t["schema"]}"."{t["table"]}"."{col_name}"'

                    # Numeric (integer, numeric, floats)
                    if (
                        data_type
                        in (
                            "integer",
                            "bigint",
                            "smallint",
                            "numeric",
                            "decimal",
                            "double precision",
                            "real",
                        )
                        or udt_name
                        in ("int2", "int4", "int8", "numeric", "float4", "float8")
                    ):
                        sql = (
                            f"SELECT MIN({full_col}) AS min_val, MAX({full_col}) AS max_val "
                            f"FROM \"{t['schema']}\".\"{t['table']}\""
                        )
                        cur.execute(sql)
                        r = cur.fetchone() or {}
                        if r.get("min_val") is not None or r.get("max_val") is not None:
                            table_profiles[col_name] = {
                                "kind": "numeric",
                                "min": r.get("min_val"),
                                "max": r.get("max_val"),
                            }
                        continue

                    # Date / timestamp
                    if (
                        data_type
                        in (
                            "date",
                            "timestamp without time zone",
                            "timestamp with time zone",
                        )
                        or "timestamp" in udt_name
                        or udt_name == "date"
                    ):
                        sql = (
                            f"SELECT MIN({full_col}) AS min_val, MAX({full_col}) AS max_val "
                            f"FROM \"{t['schema']}\".\"{t['table']}\""
                        )
                        cur.execute(sql)
                        r = cur.fetchone() or {}
                        if r.get("min_val") is not None or r.get("max_val") is not None:
                            table_profiles[col_name] = {
                                "kind": "datetime",
                                "min": r.get("min_val"),
                                "max": r.get("max_val"),
                            }
                        continue

                    # Text-like
                    if any(x in data_type for x in ["character", "text"]) or udt_name in (
                        "text",
                    ):
                        # Distinct count first
                        sql = (
                            f"SELECT COUNT(DISTINCT {full_col}) AS ndist "
                            f"FROM \"{t['schema']}\".\"{t['table']}\""
                        )
                        cur.execute(sql)
                        r = cur.fetchone() or {}
                        ndist = r.get("ndist") or 0
                        if ndist and ndist <= max_categories:
                            # Capture distinct values with frequencies
                            sql = (
                                f"SELECT {full_col} AS value, COUNT(*) AS freq "
                                f"FROM \"{t['schema']}\".\"{t['table']}\" "
                                f"GROUP BY {full_col} ORDER BY freq DESC LIMIT {max_categories}"
                            )
                            cur.execute(sql)
                            values = []
                            for row in cur.fetchall() or []:
                                values.append(
                                    {
                                        "value": row.get("value"),
                                        "freq": row.get("freq"),
                                    }
                                )
                            if values:
                                table_profiles[col_name] = {
                                    "kind": "categorical",
                                    "values": values,
                                }
                                continue
                        # Fallback: length distribution
                        sql = (
                            f"SELECT MIN(LENGTH({full_col})) AS min_len, MAX(LENGTH({full_col})) AS max_len "
                            f"FROM \"{t['schema']}\".\"{t['table']}\""
                        )
                        cur.execute(sql)
                        r = cur.fetchone() or {}
                        if r.get("min_len") is not None or r.get("max_len") is not None:
                            table_profiles[col_name] = {
                                "kind": "text",
                                "min_length": r.get("min_len"),
                                "max_length": r.get("max_len"),
                            }

    return profiles

In [ ]:
# Optional: derive per-table row targets from sample CSVs
def build_rows_per_table_override_from_sample(
    db_schema_list: List[Dict[str, Any]],
    sample_dir: str,
    base_table: str,
    target_rows_for_base: int,
 ) -> Dict[str, int]:
    """Infer rows_per_table_override from CSV row counts.

    - CSV files are expected to be named `<table>.csv` inside `sample_dir`.
    - `base_table` can be either `schema.table` or just `table` (case-insensitive).
    - All other tables are scaled proportionally to the base table's count.
    """
    import os

    counts: Dict[str, int] = {}
    table_name_to_tid: Dict[str, str] = {}  # lower(table_name) -> tid
    for t in db_schema_list:
        tid = _table_id(t)  # schema.table
        csv_name = f"{t['table']}.csv"
        csv_path = os.path.join(sample_dir, csv_name)
        if not os.path.exists(csv_path):
            continue
        df = pd.read_csv(csv_path)
        counts[tid] = len(df)
        table_name_to_tid[t["table"].lower()] = tid

    # Resolve base_table to a full table id (schema.table) if needed
    resolved_base = base_table
    if resolved_base not in counts:
        # Try matching by bare table name (case-insensitive)
        if "." not in base_table:
            base_name = base_table.lower()
            # Find all tids whose table name matches base_table
            matches = [tid for name, tid in table_name_to_tid.items() if name == base_name]
            if len(matches) == 1:
                resolved_base = matches[0]
            elif len(matches) > 1:
                raise ValueError(
                    f"Base table '{base_table}' is ambiguous; candidates: {', '.join(matches)}. "
                    "Please use 'schema.table' format.",
                )
        # If still not resolved, error out
    if resolved_base not in counts:
        raise ValueError(
            f"Base table {base_table} has no sample CSV in {sample_dir}. "
            f"Available tables with CSV: {', '.join(sorted(counts.keys()))}",
        )

    base_count = counts[resolved_base]
    if base_count <= 0:
        raise ValueError(f"Base table {resolved_base} has zero rows in sample")

    factor = target_rows_for_base / base_count

    override: Dict[str, int] = {}
    for tid, c in counts.items():
        override[tid] = max(1, int(round(c * factor)))

    return override

In [ ]:
# Table ordering and truncate helpers
def plan_table_order(db_schema_list: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Return tables in an order where parents come before children (FK-aware)."""
    table_by_id: Dict[str, Dict[str, Any]] = {}
    for t in db_schema_list:
        tid = f"{t['schema']}.{t['table']}"
        table_by_id[tid] = t

    graph: Dict[str, Set[str]] = {tid: set() for tid in table_by_id.keys()}
    indegree: Dict[str, int] = {tid: 0 for tid in table_by_id.keys()}

    for t in db_schema_list:
        child_id = f"{t['schema']}.{t['table']}"
        for fk in t["constraints"].get("foreign_keys", []):
            parent_id = f"{fk['references']['schema']}.{fk['references']['table']}"
            if parent_id == child_id:
                continue
            if parent_id in graph and child_id in graph:
                if child_id not in graph[parent_id]:
                    graph[parent_id].add(child_id)
                    indegree[child_id] += 1

    from collections import deque

    queue = deque([tid for tid, deg in indegree.items() if deg == 0])
    ordered_ids: List[str] = []

    while queue:
        tid = queue.popleft()
        ordered_ids.append(tid)
        for child in graph[tid]:
            indegree[child] -= 1
            if indegree[child] == 0:
                queue.append(child)

    if len(ordered_ids) != len(table_by_id):
        remaining = [tid for tid in table_by_id.keys() if tid not in ordered_ids]
        ordered_ids.extend(remaining)

    return [table_by_id[tid] for tid in ordered_ids]


def truncate_all_synthetic_tables(config: DBConfig, db_schema_list: List[Dict[str, Any]]) -> None:
    """Truncate all known tables in a FK-safe order (children first)."""
    ordered_tables = plan_table_order(db_schema_list)
    ordered_tables = list(reversed(ordered_tables))
    with get_connection(config) as conn:
        with conn.cursor() as cur:
            for t in ordered_tables:
                schema = t["schema"]
                table = t["table"]
                sql = f'TRUNCATE TABLE "{schema}"."{table}" RESTART IDENTITY CASCADE'
                cur.execute(sql)
        # Explicitly commit the truncation transaction so that all tables are
        # truly emptied before we start inserting synthetic rows.
        conn.commit()
    print("Truncated all user tables (FK-safe order).")

In [ ]:
# Value generation helper using Faker and optional profiles
def _generate_scalar_value(col: Dict[str, Any], stats: Optional[Dict[str, Any]] = None) -> Any:
    """Generate a single scalar value for a column using Faker + heuristics."""
    name = col["name"].lower()
    data_type = (col["data_type"] or "").lower()
    udt_name = (col["udt_name"] or "").lower()
    max_len = col.get("max_length") or 64

    if col.get("is_nullable") and random.random() < 0.05:
        return None

    if data_type == "uuid" or udt_name == "uuid":
        return str(uuid.uuid4())

    if stats:
        kind = stats.get("kind")

        if kind == "numeric" and (
            data_type
            in (
                "integer",
                "bigint",
                "smallint",
                "numeric",
                "decimal",
                "double precision",
                "real",
            )
            or udt_name in ("int2", "int4", "int8", "numeric", "float4", "float8")
):
            lo = stats.get("min")
            hi = stats.get("max")
            if lo is not None and hi is not None and lo <= hi:
                return fake.pyint(min_value=int(lo), max_value=int(hi))

        if kind == "datetime" and (
            data_type
            in (
                "date",
                "timestamp without time zone",
                "timestamp with time zone",
            )
            or "timestamp" in udt_name
            or udt_name == "date"
        ):
            return fake.date_time_between(start_date="-5y", end_date="now")

        if kind == "categorical":
            values = stats.get("values") or []
            if values:
                population = [v["value"] for v in values]
                weights = [v.get("freq", 1) for v in values]
                return random.choices(population, weights=weights, k=1)[0]

        if kind == "text" and (
            "character" in data_type
            or "text" in data_type
            or udt_name == "text"
        ):
            min_l = stats.get("min_length") or 1
            max_l = stats.get("max_length") or max_len
            if max_l < 1:
                max_l = max_len
            if min_l < 1:
                min_l = 1
            if max_l < min_l:
                max_l = min_l
            length = random.randint(min_l, max_l)
            requested = max(length, 5)
            txt = fake.text(max_nb_chars=requested)
            return txt[:length]

    def _limited_text(n: int) -> str:
        requested = max(n, 5)
        txt = fake.text(max_nb_chars=requested)
        return txt[:n]

    if "email" in name:
        return fake.email()
    if "first_name" in name:
        return fake.first_name()
    if "last_name" in name:
        return fake.last_name()
    if "full_name" in name or name == "name":
        return fake.name()
    if "phone" in name or "mobile" in name:
        return fake.phone_number()
    if "city" in name:
        return fake.city()
    if "country" in name:
        return fake.country()
    if "postcode" in name or "zipcode" in name or "zip" == name:
        return fake.postcode()

    if "timestamp" in data_type or "timestamp" in udt_name:
        return fake.date_time_between(start_date="-5y", end_date="now")
    if data_type in ("date",) or udt_name in ("date",):
        return fake.date_between(start_date="-5y", end_date="today")

    if data_type in ("boolean",) or udt_name in ("bool",):
        return fake.pybool()

    if data_type in ("integer", "bigint", "smallint") or udt_name in ("int2", "int4", "int8"):
        if data_type == "smallint" or udt_name == "int2":
            max_val = 32767
        else:
            max_val = 1_000_000
        return fake.pyint(min_value=0, max_value=max_val)

    if data_type in ("double precision", "real") or udt_name in ("float4", "float8"):
        if any(k in name for k in ["amount", "price", "total", "balance"]):
            return fake.pyfloat(min_value=0, max_value=1_000_000)
        return fake.pyfloat(min_value=0, max_value=1_000_000)

    if data_type in ("numeric", "decimal") or udt_name in ("numeric",):
        precision = col.get("numeric_precision")
        scale = col.get("numeric_scale")
        if not isinstance(precision, int) or precision <= 0:
            scale = scale if isinstance(scale, int) and scale >= 0 else 2
            return fake.pydecimal(left_digits=10, right_digits=scale, positive=True)
        if not isinstance(scale, int) or scale < 0:
            scale = 0
        if scale >= precision:
            return fake.pyint(min_value=0, max_value=10**precision - 1)
        left_digits = max(1, precision - scale)
        d = fake.pydecimal(left_digits=left_digits, right_digits=scale, positive=True)
        max_abs = Decimal(10) ** Decimal(precision - scale)
        if d >= max_abs:
            d = max_abs - (Decimal(1) / (Decimal(10) ** scale))
        return d

    if any(t in data_type for t in ("character varying", "varchar", "character", "text")) or udt_name in (
        "text",
):
        n = max_len if isinstance(max_len, int) and max_len > 0 else 128
        if any(k in name for k in ["desc", "description", "comment"]):
            return _limited_text(min(n, 200))
        return _limited_text(n)

    return _limited_text(min(max_len, 64) if isinstance(max_len, int) and max_len > 0 else 64)

In [ ]:
# Insert helpers: batch insert into Postgres
def insert_synthetic_dataset(
    config: DBConfig,
    db_schema_list: List[Dict[str, Any]],
    synthetic_data: Dict[str, List[Dict[str, Any]]],
    batch_size: int = 1000,
 ) -> None:
    """Insert synthetic_data into Postgres following FK-safe order."""
    def _truncate_string_value(value: Any, max_length: Optional[int]) -> Any:
        if value is None:
            return None
        if max_length is None:
            return value
        if not isinstance(max_length, int) or max_length <= 0:
            return value
        s = str(value)
        if len(s) <= max_length:
            return value
        return s[:max_length]

    ordered_tables = plan_table_order(db_schema_list)

    with get_connection(config) as conn:
        with conn.cursor() as cur:
            for t in ordered_tables:
                tid = _table_id(t)
                rows = synthetic_data.get(tid) or []
                if not rows:
                    continue

                columns = [c["name"] for c in t["columns"]]
                col_max = {c["name"]: c.get("max_length") for c in t["columns"]}

                placeholders = ", ".join(["%s"] * len(columns))
                col_list_sql = ", ".join(f'"{c}"' for c in columns)
                sql = (
                    f'INSERT INTO "{t["schema"]}"."{t["table"]}" '
                    f'({col_list_sql}) VALUES ({placeholders})'
                )

                batch: List[tuple] = []
                for row in rows:
                    vals = []
                    for c in columns:
                        v = row.get(c)
                        if isinstance(v, str):
                            v = _truncate_string_value(v, col_max.get(c))
                        vals.append(v)
                    batch.append(tuple(vals))
                    if len(batch) >= batch_size:
                        execute_batch(cur, sql, batch)
                        batch = []
                if batch:
                    execute_batch(cur, sql, batch)
        conn.commit()


# Validation helpers: check FK referential integrity
def validate_referential_integrity(config: DBConfig, db_schema_list: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Validate FK constraints by checking for orphaned child rows."""
    results: Dict[str, Any] = {}

    with get_connection(config) as conn:
        with conn.cursor() as cur:
            for t in db_schema_list:
                tid = _table_id(t)
                for fk in t["constraints"].get("foreign_keys", []):
                    child_cols = fk["columns"]
                    ref = fk["references"]
                    parent_schema = ref["schema"]
                    parent_table = ref["table"]
                    parent_cols = ref["columns"]

                    join_conditions = " AND ".join(
                        f'c."{c}" = p."{p}"' for c, p in zip(child_cols, parent_cols)
                    )
                    where_null = " AND ".join(f'c."{c}" IS NOT NULL' for c in child_cols)

                    sql = f"""
                    SELECT count(*) AS orphan_count
                    FROM "{t['schema']}"."{t['table']}" c
                    LEFT JOIN "{parent_schema}"."{parent_table}" p
                      ON {join_conditions}
                    WHERE {where_null} AND p."{parent_cols[0]}" IS NULL
                    """

                    cur.execute(sql)
                    row = cur.fetchone()
                    if isinstance(row, dict):
                        count = row.get("orphan_count", 0)
                    else:
                        count = row[0]
                    if count > 0:
                        key = f"{tid}:{','.join(child_cols)}->{parent_schema}.{parent_table}({','.join(parent_cols)})"
                        results[key] = {"orphan_count": count}

    return results

In [ ]:
# Batched pipeline runner: orchestrates schema, profiling, generation, insert, and validation
def run_synthetic_pipeline_batched(
    config: DBConfig,
    target_rows_per_table: int = 100,
    rows_per_table_override: Optional[Dict[str, int]] = None,
    batch_size: int = 10_000,
    use_existing_data_profile: bool = False,
    sample_dir: Optional[str] = None,
    truncate_before_insert: bool = True,
    validate_fk: bool = True,
    seed: Optional[int] = None,
 ) -> Dict[str, Any]:
    """Batched version of the pipeline: generates and inserts in chunks."""
    if seed is not None:
        random.seed(seed)
        Faker.seed(seed)

    print("[pipeline] Discovering schema and planning FK-safe order...")
    db_schema_list = fetch_db_schema(config)
    ordered_tables = plan_table_order(db_schema_list)

    profiles: Dict[str, Dict[str, Dict[str, Any]]] = {}
    if use_existing_data_profile:
        existing_stats = profile_existing_data(config, db_schema_list)
        for tid, cols in existing_stats.items():
            profiles.setdefault(tid, {}).update(cols)

    if sample_dir is not None:
        sample_stats = profile_sample_data(db_schema_list, sample_dir=sample_dir)
        for tid, cols in sample_stats.items():
            profiles.setdefault(tid, {}).update(cols)

    if truncate_before_insert:
        print("[pipeline] Truncating all user tables (FK-safe order)...")
        truncate_all_synthetic_tables(config, db_schema_list)

    accumulated_counts: Dict[str, int] = {}
    parent_rows_cache: Dict[str, List[Dict[str, Any]]] = {}
    pk_counters: Dict[str, Dict[str, int]] = {}

    unique_cols_map: Dict[str, set] = {}
    for _t in db_schema_list:
        _tid = _table_id(_t)
        _pk_cols: List[str] = []
        for _pk in _t["constraints"].get("primary_keys", []):
            _pk_cols.extend(_pk["columns"])
        _unique_cols = set(_pk_cols)
        for _uq in _t["constraints"].get("uniques", []):
            for _c in _uq.get("columns", []):
                _unique_cols.add(_c)
        unique_cols_map[_tid] = _unique_cols

    used_unique_values: Dict[str, Dict[str, set]] = {}

    print("[pipeline] Starting batched synthetic load...")

    for t in ordered_tables:
        tid = _table_id(t)
        cols = t["columns"]
        constraints = t["constraints"]

        pk_counters.setdefault(tid, {})
        used_unique_values.setdefault(tid, {})
        table_unique_cols = unique_cols_map.get(tid, set())

        target_rows = (
            rows_per_table_override.get(tid, target_rows_per_table)
            if rows_per_table_override
            else target_rows_per_table
)

        total_batches = max(1, math.ceil(target_rows / batch_size))

        print(
            f"[pipeline] Table {tid}: target_rows={target_rows}, batch_size={batch_size}, total_batches={total_batches}",
)

        accumulated_counts[tid] = 0

        for _ in range(total_batches):
            remaining = target_rows - accumulated_counts[tid]
            if remaining <= 0:
                break
            this_batch = min(batch_size, remaining)

            batch_rows: List[Dict[str, Any]] = []

            fk_constraints = constraints.get("foreign_keys", [])
            fk_child_cols = {c for fk in fk_constraints for c in fk["columns"]}
            pk_cols: List[str] = []
            for pk in constraints.get("primary_keys", []):
                pk_cols.extend(pk["columns"])

            for _ in range(this_batch):
                row: Dict[str, Any] = {}

                for fk in fk_constraints:
                    parent_tid = f"{fk['references']['schema']}.{fk['references']['table']}"
                    parent_rows = parent_rows_cache.get(parent_tid) or []
                    if not parent_rows:
                        continue
                    parent_row = random.choice(parent_rows)
                    for child_col, parent_col in zip(
                        fk["columns"], fk["references"]["columns"]
):
                        row[child_col] = parent_row[parent_col]

                for col in cols:
                    cname = col["name"]
                    if cname in row:
                        continue

                    data_type = (col.get("data_type") or "").lower()
                    udt_name = (col.get("udt_name") or "").lower()

                    # Integer primary key columns: generate sequential IDs
                    # per table/column so we never hit duplicate PKs, even in
                    # batched mode. This is generic and does not depend on
                    # specific table names.
                    if cname in pk_cols and data_type in (
                        "integer",
                        "bigint",
                        "smallint",
):
                        c = pk_counters[tid].get(cname, 0) + 1
                        pk_counters[tid][cname] = c
                        row[cname] = c
                        continue

                    # Text/varchar primary key columns: generate a deterministic
                    # pattern-based ID (e.g. BRANCH_ID_000001) instead of relying
                    # on Faker. This guarantees no duplicate PKs for text IDs like
                    # branch_id, customer_id, etc.
                    if cname in pk_cols and (
                        "char" in data_type
                        or "text" in data_type
                        or udt_name == "text"
):
                        c = pk_counters[tid].get(cname, 0) + 1
                        pk_counters[tid][cname] = c
                        base = cname.upper()[:10] or "COL"
                        value = f"{base}_{c:06d}"
                        max_l = col.get("max_length")
                        if isinstance(max_l, int) and max_l > 0:
                            value = value[:max_l]
                        row[cname] = value
                        used_for_table = used_unique_values[tid].setdefault(cname, set())
                        used_for_table.add(value)
                        continue

                    # For FK columns, only use values copied from parents.
                    if cname in fk_child_cols:
                        continue

                    col_stats = None
                    if cname not in table_unique_cols:
                        col_stats = profiles.get(tid, {}).get(cname)

                    value = _generate_scalar_value(col, stats=col_stats)

                    if cname in table_unique_cols:
                        used_for_table = used_unique_values[tid].setdefault(cname, set())
                        attempts = 0
                        max_attempts = 10
                        while (value is None or value in used_for_table) and attempts < max_attempts:
                            value = _generate_scalar_value(col, stats=None)
                            attempts += 1

                        if value is None or value in used_for_table:
                            data_type = (col.get("data_type") or "").lower()
                            udt = (col.get("udt_name") or "").lower()
                            suffix = len(used_for_table) + 1
                            if data_type in ("integer", "bigint", "smallint") or udt in (
                                "int2",
                                "int4",
                                "int8",
):
                                value = suffix
                            else:
                                base = cname.upper()[:10] or "COL"
                                value = f"{base}_{suffix:06d}"
                                max_l = col.get("max_length")
                                if isinstance(max_l, int) and max_l > 0:
                                    value = value[:max_l]

                        used_for_table.add(value)

                    row[cname] = value

                batch_rows.append(row)

            synthetic_data_single = {tid: batch_rows}
            insert_synthetic_dataset(config, [t], synthetic_data_single, batch_size=batch_size)
            accumulated_counts[tid] += len(batch_rows)

            parent_rows_cache[tid] = parent_rows_cache.get(tid, []) + batch_rows

            print(
                f"[pipeline] Table {tid}: batch inserted {len(batch_rows)} rows, "
                f"cumulative={accumulated_counts[tid]}/{target_rows}",
            )

    fk_result = None
    if validate_fk:
        fk_result = validate_referential_integrity(config, db_schema_list)

    return {
        "schema": db_schema_list,
        "profiles": profiles,
        "row_counts": accumulated_counts,
        "fk_violations": fk_result,
    }

In [ ]:
# 1. Configure environment (no src imports needed now)
import os

from dotenv import load_dotenv

# Load environment variables from .env at project root
project_root = os.path.abspath(os.getcwd())
load_dotenv(dotenv_path=os.path.join(project_root, ".env"))

cfg = DBConfig(
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "5432")),
    dbname=os.getenv("DB_NAME", "postgres"),
    user=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD", ""),
)

target_rows = int(os.getenv("TARGET_ROWS_PER_TABLE", "100000"))
batch_size = int(os.getenv("BATCH_SIZE", "20000"))
use_existing_data_profile = os.getenv("USE_EXISTING_DATA_PROFILE", "false").lower() in ("1", "true", "yes")
sample_dir = os.getenv("SAMPLE_DIR") or None
base_table_for_override = os.getenv("BASE_TABLE_FOR_SAMPLE_OVERRIDE") or None

print("DB config:", cfg)
print("TARGET_ROWS_PER_TABLE:", target_rows)
print("BATCH_SIZE:", batch_size)
print("USE_EXISTING_DATA_PROFILE:", use_existing_data_profile)
print("SAMPLE_DIR:", sample_dir)
print("BASE_TABLE_FOR_SAMPLE_OVERRIDE:", base_table_for_override)

In [ ]:
# 1. Configure environment (no src imports needed now)
import os

from dotenv import load_dotenv

# Load environment variables from .env at project root
project_root = os.path.abspath(os.getcwd())
load_dotenv(dotenv_path=os.path.join(project_root, ".env"))

cfg = DBConfig(
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "5432")),
    dbname=os.getenv("DB_NAME", "postgres"),
    user=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD", ""),
)

target_rows = int(os.getenv("TARGET_ROWS_PER_TABLE", "100000"))
batch_size = int(os.getenv("BATCH_SIZE", "20000"))
use_existing_data_profile = os.getenv("USE_EXISTING_DATA_PROFILE", "false").lower() in ("1", "true", "yes")
sample_dir = os.getenv("SAMPLE_DIR") or None

print("DB config:", cfg)
print("TARGET_ROWS_PER_TABLE:", target_rows)
print("BATCH_SIZE:", batch_size)
print("USE_EXISTING_DATA_PROFILE:", use_existing_data_profile)
print("SAMPLE_DIR:", sample_dir)

## Step 1 – Discover schema and FK-safe order

This step uses the **local helper functions** `fetch_db_schema` and
`plan_table_order` defined above to:

- Read all user tables, columns, and constraints from `information_schema`.
- Derive a generic representation (no hard-coded table names).
- Compute an FK-safe table order where parents come before children.

In [ ]:
# 2. Discover schema and FK-safe table order using src.schematic helpers
db_schema = fetch_db_schema(cfg)
print(f"Discovered {len(db_schema)} tables in DB '{cfg.dbname}'.")
for t in db_schema:
    print(" -", f"{t['schema']}.{t['table']}", "(" + str(len(t['columns'])) + " columns)")
    
ordered_tables = plan_table_order(db_schema)
print("\nFK-safe table order (parents before children):")
for t in ordered_tables:
    print(" -", f"{t['schema']}.{t['table']}")

## Step 2 – Profile existing data and/or sample CSVs

This step uses the **local profiling helpers** to build simple
column-level statistics that guide Faker (optional but recommended):

- `profile_existing_data` – profiles live DB data (min/max, categories, text lengths).
- `profile_sample_data` – profiles CSVs from `SAMPLE_DIR` (e.g. `sample_data`, `dealer_db`).
- The resulting profiles are merged and passed into the generator so synthetic values stay realistic.

In [ ]:
# 3. Build column profiles from DB and sample CSVs (optional but recommended)
profiles = {}
if use_existing_data_profile:
    print("Profiling existing DB data (ranges, categories, lengths)...")
    existing_stats = profile_existing_data(cfg, db_schema)
    for tid, cols in existing_stats.items():
        profiles.setdefault(tid, {}).update(cols)

if sample_dir:
    print(f"Profiling sample CSVs in folder: {sample_dir} ...")
    sample_stats = profile_sample_data(db_schema, sample_dir=sample_dir)
    for tid, cols in sample_stats.items():
        profiles.setdefault(tid, {}).update(cols)

print("\nProfiled tables (tid -> num columns with stats):")
for tid, cols in profiles.items():
    print(" -", tid, ":", len(cols), "columns")

## Step 3 – Run batched synthetic pipeline and review results

This step calls the **local** `run_synthetic_pipeline_batched` function
defined above (no imports from `src`).

Key points:
- Uses the same `DBConfig` and `.env` parameters you set earlier.
- Leverages the `profiles` we just built (when available) to guide realistic value generation.
- Inserts data in FK-safe order and enforces uniqueness for primary/unique keys across batches.
- Returns final row counts per table and a summary of FK validation results.

In [ ]:
# 4. Run the batched synthetic pipeline using local implementation
import time

# Optionally derive per-table row targets from sample CSVs if configured
rows_per_table_override = None
if sample_dir and base_table_for_override:
    rows_per_table_override = build_rows_per_table_override_from_sample(
        db_schema_list=db_schema,
        sample_dir=sample_dir,
        base_table=base_table_for_override,
        target_rows_for_base=target_rows,
    )
    print(f"Using rows_per_table_override based on sample data (base_table={base_table_for_override}).")
else:
    print("Using uniform TARGET_ROWS_PER_TABLE for all tables.")

row_counts_result = run_synthetic_pipeline_batched(
    config=cfg,
    target_rows_per_table=target_rows,
    rows_per_table_override=rows_per_table_override,
    batch_size=batch_size,
    use_existing_data_profile=use_existing_data_profile,
    sample_dir=sample_dir,
    truncate_before_insert=True,
    validate_fk=True,
    seed=42,
 )

row_counts = row_counts_result.get("row_counts", {})
fk_violations = row_counts_result.get("fk_violations", {})

print("\nFinal row counts per table:")
for t_name, cnt in sorted(row_counts.items()):
    print(f" - {t_name}: {cnt:,} rows")

print("\nFK validation results:")
if not fk_violations:
    print(" - No FK violations detected ✅")
else:
    for rel, info in fk_violations.items():
        print(f" - {rel}: {info['orphan_count']} orphan rows")

#21:29 min